# Fine-Tuning a Pre-trained CNN

**Case Study:** Classifying the digits 3 and 8 using 5000 images from the built-in MNIST dataset.


## 1. Import libraries

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

## 2. Load MNIST and select 5000 images

In [2]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Select digits 3 and 8 only
train_mask = (y_train == 3) | (y_train == 8)
test_mask = (y_test == 3) | (y_test == 8)

x_train = x_train[train_mask][:5000]
y_train = y_train[train_mask][:5000]

x_test = x_test[test_mask]
y_test = y_test[test_mask]

# New labels: 3 = 0 and 8 = 1
y_train = (y_train == 8).astype('int32')
y_test = (y_test == 8).astype('int32')

print('Training data:', x_train.shape)
print('Testing data:', x_test.shape)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
Training data: (5000, 28, 28)
Testing data: (1984, 28, 28)


## 3. Convert grayscale images to RGB

In [3]:
x_train = tf.expand_dims(x_train, axis=-1)
x_test = tf.expand_dims(x_test, axis=-1)

x_train = tf.image.grayscale_to_rgb(x_train)
x_test = tf.image.grayscale_to_rgb(x_test)

## 4. Load and modify MobileNetV2

In [4]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(96, 96, 3)
)

base_model.trainable = False

model = models.Sequential([
    layers.Input(shape=(28, 28, 3)),
    layers.Resizing(96, 96),
    layers.Rescaling(1./127.5, offset=-1),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 5s 1us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resizing (Resizing)             │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_96             │ (None, 3, 3, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

## 5. Train the new classifier

In [5]:
model.fit(
    x_train,
    y_train,
    epochs=3,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/3
125/125 ━━━━━━━━━━━━━━━━━━━━ 13s 82ms/step - accuracy: 0.9415 - loss: 0.1910 - val_accuracy: 0.9920 - val_loss: 0.0663
Epoch 2/3
125/125 ━━━━━━━━━━━━━━━━━━━━ 11s 85ms/step - accuracy: 0.9835 - loss: 0.0696 - val_accuracy: 0.9970 - val_loss: 0.0398
Epoch 3/3
125/125 ━━━━━━━━━━━━━━━━━━━━ 9s 74ms/step - accuracy: 0.9877 - loss: 0.0493 - val_accuracy: 0.9980 - val_loss: 0.0300


## 6. Fine-tuning

In [6]:
base_model.trainable = True

# Train only the last 10 layers
for layer in base_model.layers[:-10]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.fit(
    x_train,
    y_train,
    epochs=2,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/2
125/125 ━━━━━━━━━━━━━━━━━━━━ 18s 90ms/step - accuracy: 0.9675 - loss: 0.0893 - val_accuracy: 0.9990 - val_loss: 0.0223
Epoch 2/2
125/125 ━━━━━━━━━━━━━━━━━━━━ 10s 82ms/step - accuracy: 0.9845 - loss: 0.0494 - val_accuracy: 0.9980 - val_loss: 0.0194


## 7. Evaluate the model

In [7]:
loss, accuracy = model.evaluate(x_test, y_test)

print('Test Loss:', loss)
print('Test Accuracy:', accuracy)

62/62 ━━━━━━━━━━━━━━━━━━━━ 4s 59ms/step - accuracy: 0.9909 - loss: 0.0328
Test Loss: 0.03280845284461975
Test Accuracy: 0.9909273982048035


## Conclusion

MobileNetV2 was used as a pre-trained CNN. Its original classification layer was removed and replaced with a binary classifier. First, the pre-trained layers were frozen, then the last 10 layers were fine-tuned using a small learning rate.